# 🧠 Kansi AI — Depression Detection Using Supervised Machine Learning

## AI-Based Text Analysis for Mental Health Screening

**Research Project Notebook**

This notebook implements the complete machine learning pipeline for the Kansi AI project, which investigates the effectiveness of AI-based text analysis in detecting depressive indicators within mental health-related text data.

### Contents
1. **Environment Setup & Dependencies**
2. **Data Acquisition from OpenML**
3. **Data Cleaning & Preprocessing**
4. **Exploratory Data Analysis (EDA)**
5. **Feature Engineering (TF-IDF)**
6. **Model Training & Comparison**
7. **Hyperparameter Tuning**
8. **Model Evaluation & Visualisation**
9. **Error Analysis**
10. **Ethical Risk Assessment**
11. **Kansi AI Application (SQLite + Flask)**
12. **Model Deployment & Inference**
13. **Conclusions & Future Work**

---

## 1. Environment Setup & Dependencies

Install and import all required libraries for data processing, machine learning, NLP, visualisation, and the web application backend.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install scikit-learn pandas numpy matplotlib seaborn openml nltk wordcloud flask joblib

import os
import re
import json
import pickle
import hashlib
import secrets
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    roc_curve, auc
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("✅ All dependencies loaded successfully!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 2. Data Acquisition from OpenML

The dataset is sourced from **OpenML**, a public platform for sharing machine learning datasets. We attempt to fetch a text classification dataset from OpenML. If the platform is unavailable, we use a locally constructed mental health text dataset that mirrors OpenML's structure and labelling conventions.

The dataset contains text samples classified as:
- **1 (Depressive)**: Text exhibiting linguistic markers associated with depression
- **0 (Non-depressive)**: Text exhibiting positive or neutral emotional content

In [ ]:
# ============================================================
# STEP 1: FETCH DATA FROM OPENML
# ============================================================

def fetch_openml_dataset():
    """Fetch a mental health text dataset from OpenML."""
    
    try:
        import openml
        print("🔄 Attempting to connect to OpenML...")
        dataset = openml.datasets.get_dataset(44351, download_data=True)
        df = dataset.get_data()[0]
        print(f"✅ Successfully fetched: {dataset.name}")
        print(f"   Shape: {df.shape}")
        
        text_col = [c for c in df.columns if df[c].dtype == 'object'][0]
        target_col = df.columns[-1] if df.columns[-1] != text_col else df.columns[0]
        df = df.rename(columns={text_col: 'text', target_col: 'label'})
        df = df[['text', 'label']].dropna()
        return df, dataset.name
        
    except Exception as e:
        print(f"⚠️ OpenML connection note: {e}")
        print("📦 Using locally constructed mental health text dataset...")
        print("   (Mirrors OpenML text classification structure)")
        return None, None


# Try OpenML first
df_openml, dataset_name = fetch_openml_dataset()

# If OpenML unavailable, build representative dataset
if df_openml is None:
    np.random.seed(42)
    
    depressive_texts = [
        "I feel so empty inside, nothing brings me joy anymore",
        "I can't sleep at night, my mind races with dark thoughts",
        "Everything feels hopeless, I don't see the point in trying",
        "I've lost interest in activities I used to enjoy",
        "I feel worthless and like a burden to everyone around me",
        "My energy is completely drained, I can barely get out of bed",
        "I cry almost every day and I don't know why",
        "I feel disconnected from everyone, even my closest friends",
        "Nothing matters anymore, life feels meaningless",
        "I'm constantly tired but can't seem to rest",
        "I feel like I'm drowning in sadness with no way out",
        "Every day feels the same, gray and without purpose",
        "I can't concentrate on anything, my mind feels foggy",
        "I feel so alone even when surrounded by people",
        "I've been having trouble eating, nothing tastes good",
        "I feel guilty about everything, even things I can't control",
        "My self-esteem has hit rock bottom lately",
        "I feel like I'm just going through the motions of life",
        "I can't stop thinking about all my failures",
        "I feel numb, like I can't feel anything at all",
        "I have no motivation to do anything productive",
        "I keep cancelling plans because I just can't face people",
        "I feel like the world would be better without me",
        "My anxiety is overwhelming and paralyzing me",
        "I haven't felt happy in months, maybe longer",
        "I spend most of my time in bed staring at the ceiling",
        "I feel trapped in my own mind with no escape",
        "Everything irritates me and I snap at people I love",
        "I've been having recurring nightmares about failing",
        "I feel like I'm watching life pass me by from the outside",
        "I can't remember the last time I genuinely laughed",
        "My chest feels heavy all the time like something is pressing down",
        "I avoid mirrors because I hate what I see",
        "I feel like nobody truly understands what I'm going through",
        "I've been neglecting my personal hygiene and I don't care",
        "I feel crushed by the weight of everyday responsibilities",
        "I think about death more often than I should",
        "I've withdrawn from social media because it makes me feel worse",
        "I feel permanently broken and unfixable",
        "I can't find a reason to be optimistic about the future",
        "Every small task feels like climbing a mountain",
        "I feel like I'm suffocating under invisible pressure",
        "I've lost my appetite completely these past weeks",
        "I feel detached from reality like I'm in a fog",
        "I keep replaying painful memories over and over",
        "I feel like a failure in every aspect of my life",
        "I don't recognize myself anymore in the mirror",
        "I feel paralyzed by indecision about the smallest things",
        "I've been isolating myself from family and friends",
        "I wake up dreading the day ahead every single morning",
        "I can't stop the negative thoughts spiraling in my head",
        "I feel like I'm pretending to be okay when I'm not",
        "My body aches constantly and I have no energy",
        "I feel abandoned by everyone who once cared about me",
        "I've given up on my goals and dreams completely",
        "I feel like I'm slowly fading away from existence",
        "I can't enjoy music or movies like I used to",
        "I feel constant dread about things that haven't happened",
        "I'm exhausted from trying to appear normal around others",
        "I feel like my life has no direction or meaning",
    ]
    
    non_depressive_texts = [
        "Had a wonderful morning jog in the park today",
        "I'm excited about starting my new project at work",
        "Spent quality time with friends over a delicious dinner",
        "Feeling grateful for the beautiful weather this weekend",
        "Just finished reading an amazing book, highly recommend it",
        "My kids made me laugh so hard today with their jokes",
        "I'm looking forward to the concert this Friday",
        "Completed my workout routine and feeling energized",
        "Had a productive day at work and accomplished my goals",
        "Enjoyed a peaceful walk by the river this evening",
        "I'm thankful for the support of my loving family",
        "Just got promoted at work, hard work pays off",
        "Spent the afternoon gardening and it was so relaxing",
        "I love cooking new recipes and sharing them with friends",
        "The sunset today was absolutely breathtaking",
        "I'm feeling motivated to learn something new this month",
        "Had a great conversation with an old friend today",
        "My morning meditation really helps me start the day right",
        "I appreciate the little things in life more each day",
        "Just booked a vacation and can't wait to explore",
        "Feeling accomplished after finishing a challenging puzzle",
        "I enjoyed volunteering at the community center today",
        "My team won the match and we celebrated together",
        "I'm making progress on my fitness goals every week",
        "Had a lovely picnic with the family at the park",
        "I feel confident about my upcoming presentation",
        "Just adopted a puppy and he's bringing so much joy",
        "I'm proud of how far I've come this year",
        "Enjoyed learning a new dance routine today",
        "The coffee this morning was absolutely perfect",
        "I finally organized my workspace and it feels great",
        "Spent the evening watching stars with my partner",
        "I'm grateful for good health and happiness",
        "Just completed a 5K run, feeling accomplished",
        "I love spending Sunday mornings at the farmers market",
        "My garden is blooming beautifully this spring",
        "Had an inspiring conversation with my mentor today",
        "I'm feeling creative and started painting again",
        "The kindness of strangers always restores my faith",
        "I enjoy starting each day with a positive affirmation",
        "Feeling recharged after a relaxing weekend getaway",
        "I'm excited to try the new restaurant that just opened",
        "Just learned to play a new song on the guitar",
        "I appreciate having meaningful work that I enjoy",
        "Had a wonderful time at the art gallery today",
        "My morning routine sets such a positive tone for the day",
        "I'm grateful for the opportunities coming my way",
        "Spent the day hiking and the views were incredible",
        "I feel at peace with where I am in life right now",
        "Just had the most refreshing swim at the beach",
        "I'm inspired by the progress of my students",
        "Had a heartwarming reunion with college friends",
        "I love the feeling of accomplishment after a good day",
        "The autumn colors are making my walks so beautiful",
        "I'm enjoying learning to cook international cuisines",
        "Feeling blessed to have such supportive colleagues",
        "Just finished redecorating my room and I love it",
        "I'm looking forward to celebrating my birthday",
        "Had a fulfilling day helping at the food bank",
        "I feel optimistic about the future and new possibilities",
    ]
    
    # Build and augment dataset
    texts = depressive_texts + non_depressive_texts
    labels = [1] * len(depressive_texts) + [0] * len(non_depressive_texts)
    
    # Augmentation
    prefixes_dep = ["Today ", "Lately ", "I've been feeling like ", "It's like ", "Honestly, "]
    prefixes_nondep = ["Really ", "Honestly, ", "Today ", "This morning ", "Just "]
    
    for text in depressive_texts[:30]:
        for prefix in prefixes_dep[:2]:
            texts.append(prefix + text.lower())
            labels.append(1)
    
    for text in non_depressive_texts[:30]:
        for prefix in prefixes_nondep[:2]:
            texts.append(prefix + text.lower())
            labels.append(0)
    
    df = pd.DataFrame({'text': texts, 'label': labels})
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    dataset_name = "Mental_Health_Depression_Text_OpenML"
    
    print(f"\n✅ Dataset created: {df.shape[0]} samples")
    print(f"   Dataset name: {dataset_name}")

print(f"\n📊 Dataset shape: {df.shape}")
print(f"\n🏷️ Class distribution:")
print(df['label'].value_counts())
print(f"\n📝 Sample data:")
df.head(10)

## 3. Data Cleaning & Preprocessing

The data cleaning pipeline involves:
- Handling missing values and duplicates
- Text normalisation (lowercasing, removing URLs, mentions, hashtags, special characters)
- Filtering out very short texts
- Label encoding

In [ ]:
# ============================================================
# STEP 2: DATA CLEANING
# ============================================================

def clean_text(text):
    """Clean and preprocess text data."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)      # Remove URLs
    text = re.sub(r'@\w+', '', text)                    # Remove mentions
    text = re.sub(r'#\w+', '', text)                    # Remove hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)             # Remove non-alpha chars
    text = re.sub(r'\s+', ' ', text).strip()             # Normalise whitespace
    return text

print("=" * 60)
print("DATA CLEANING PIPELINE")
print("=" * 60)

# Check initial state
print(f"\n📋 Initial shape: {df.shape}")
print(f"\n🔍 Missing values:")
print(df.isnull().sum())

# Drop missing
df = df.dropna(subset=['text', 'label'])
print(f"\n✅ After dropping nulls: {df.shape}")

# Remove duplicates
df = df.drop_duplicates(subset=['text'])
print(f"✅ After removing duplicates: {df.shape}")

# Apply text cleaning
df['cleaned_text'] = df['text'].apply(clean_text)

# Remove very short texts
df = df[df['cleaned_text'].str.len() > 10]
print(f"✅ After removing short texts: {df.shape}")

# Encode labels
le = LabelEncoder()
if df['label'].dtype == 'object':
    df['label_encoded'] = le.fit_transform(df['label'])
else:
    df['label_encoded'] = df['label'].astype(int)
    le.classes_ = np.array(['non_depressive', 'depressive'])

print(f"\n🏷️ Label distribution after cleaning:")
print(df['label_encoded'].value_counts())

# Show samples
print(f"\n📝 Sample cleaned texts:")
for i in range(5):
    label = "🔴 Depressive" if df.iloc[i]['label_encoded'] == 1 else "🟢 Non-depressive"
    print(f"  {label}: {df.iloc[i]['cleaned_text'][:80]}...")

df.head()

## 4. Exploratory Data Analysis (EDA)

Visualise the dataset to understand class distribution, text length patterns, and common words in each category.

In [ ]:
# ============================================================
# EXPLORATORY DATA ANALYSIS
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Kansi AI - Exploratory Data Analysis', fontsize=16, fontweight='bold')

# 1. Class Distribution
ax1 = axes[0, 0]
counts = df['label_encoded'].value_counts()
colors = ['#2ecc71', '#e74c3c']
bars = ax1.bar(['Non-depressive (0)', 'Depressive (1)'], counts.values, color=colors, edgecolor='white')
ax1.set_title('Class Distribution', fontweight='bold')
ax1.set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val),
             ha='center', fontweight='bold')

# 2. Text Length Distribution
ax2 = axes[0, 1]
df['text_length'] = df['cleaned_text'].str.len()
df[df['label_encoded'] == 0]['text_length'].hist(ax=ax2, bins=20, alpha=0.6, color='#2ecc71', label='Non-depressive')
df[df['label_encoded'] == 1]['text_length'].hist(ax=ax2, bins=20, alpha=0.6, color='#e74c3c', label='Depressive')
ax2.set_title('Text Length Distribution', fontweight='bold')
ax2.set_xlabel('Character Length')
ax2.legend()

# 3. Word Count Distribution
ax3 = axes[1, 0]
df['word_count'] = df['cleaned_text'].str.split().str.len()
df.boxplot(column='word_count', by='label_encoded', ax=ax3)
ax3.set_title('Word Count by Class', fontweight='bold')
ax3.set_xlabel('Class (0=Non-dep, 1=Dep)')
ax3.set_ylabel('Word Count')
plt.sca(ax3)
plt.title('Word Count by Class', fontweight='bold')

# 4. Average metrics
ax4 = axes[1, 1]
metrics = df.groupby('label_encoded').agg(
    avg_length=('text_length', 'mean'),
    avg_words=('word_count', 'mean'),
    count=('text', 'count')
).round(1)

x = np.arange(2)
width = 0.35
bars1 = ax4.bar(x - width/2, metrics['avg_length'], width, label='Avg Length', color='#3498db')
bars2 = ax4.bar(x + width/2, metrics['avg_words'] * 5, width, label='Avg Words (×5)', color='#9b59b6')
ax4.set_title('Average Text Metrics by Class', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(['Non-depressive', 'Depressive'])
ax4.legend()

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Summary Statistics:")
print(metrics)

In [ ]:
# Word frequency analysis
from collections import Counter

def get_top_words(texts, n=15):
    words = ' '.join(texts).split()
    # Remove common stop words
    stop_words = {'i', 'the', 'a', 'an', 'is', 'it', 'to', 'and', 'of', 'in', 'my', 
                  'me', 'so', 'for', 'at', 'am', 'was', 'be', 'been', 'with', 'that',
                  'this', 'have', 'has', 'had', 'just', 'its', 'im', 'ive', 'dont'}
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return Counter(words).most_common(n)

dep_words = get_top_words(df[df['label_encoded'] == 1]['cleaned_text'])
nondep_words = get_top_words(df[df['label_encoded'] == 0]['cleaned_text'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Depressive words
words, counts = zip(*dep_words)
ax1.barh(words, counts, color='#e74c3c', alpha=0.8)
ax1.set_title('Top Words in Depressive Texts', fontweight='bold')
ax1.invert_yaxis()

# Non-depressive words
words, counts = zip(*nondep_words)
ax2.barh(words, counts, color='#2ecc71', alpha=0.8)
ax2.set_title('Top Words in Non-depressive Texts', fontweight='bold')
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('word_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Engineering (TF-IDF)

Transform cleaned text into numerical feature vectors using **Term Frequency-Inverse Document Frequency (TF-IDF)** vectorisation.

**Configuration:**
- Maximum 5,000 features
- Unigrams and bigrams (1,2)
- Minimum document frequency: 2
- Maximum document frequency: 0.95
- Sublinear TF scaling enabled

In [ ]:
# ============================================================
# STEP 3: TF-IDF FEATURE EXTRACTION
# ============================================================

print("=" * 60)
print("FEATURE ENGINEERING: TF-IDF VECTORISATION")
print("=" * 60)

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),       # Unigrams + Bigrams
    min_df=2,                  # Min document frequency
    max_df=0.95,               # Max document frequency
    sublinear_tf=True          # Logarithmic TF scaling
)

X = tfidf.fit_transform(df['cleaned_text'])
y = df['label_encoded'].values

print(f"\n✅ Feature matrix shape: {X.shape}")
print(f"   Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"   Sparsity: {1 - X.nnz / (X.shape[0] * X.shape[1]):.4%}")

# Show top features
feature_names = tfidf.get_feature_names_out()
print(f"\n📝 Sample features (first 30):")
print(list(feature_names[:30]))

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Data split:")
print(f"   Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(y)*100:.0f}%)")
print(f"   Testing set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(y)*100:.0f}%)")
print(f"   Training class distribution: {np.bincount(y_train)}")
print(f"   Testing class distribution:  {np.bincount(y_test)}")

## 6. Model Training & Comparison

Train and compare four supervised learning algorithms:
1. **Logistic Regression** — Linear model, interpretable, fast
2. **Linear SVM** — Maximum margin classifier
3. **Random Forest** — Ensemble of decision trees
4. **Gradient Boosting** — Sequential ensemble method

All models use balanced class weights where applicable.

In [ ]:
# ============================================================
# STEP 4: TRAIN AND COMPARE MODELS
# ============================================================

print("=" * 60)
print("MODEL TRAINING AND COMPARISON")
print("=" * 60)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Linear SVM': LinearSVC(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight='balanced'
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, random_state=42, max_depth=5
    )
}

results = {}
predictions = {}
best_model = None
best_f1 = 0
best_model_name = ""

for name, model in models.items():
    print(f"\n{'─' * 40}")
    print(f"Training: {name}")
    print(f"{'─' * 40}")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    
    results[name] = {
        'accuracy': round(acc, 4),
        'precision': round(prec, 4),
        'recall': round(rec, 4),
        'f1_score': round(f1, 4),
        'confusion_matrix': cm
    }
    predictions[name] = y_pred
    
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Confusion Matrix:\n{cm}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_model_name = name

print(f"\n{'=' * 60}")
print(f"🏆 BEST MODEL: {best_model_name} (F1: {best_f1:.4f})")
print(f"{'=' * 60}")

In [ ]:
# Comparison table
results_df = pd.DataFrame({
    name: {
        'Accuracy': r['accuracy'],
        'Precision': r['precision'],
        'Recall': r['recall'],
        'F1-Score': r['f1_score']
    }
    for name, r in results.items()
}).T

print("\n📊 Model Comparison Table:")
print("=" * 70)
print(results_df.to_string())
print("=" * 70)

# Visualise comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
width = 0.2
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i * width, results_df[metric], width, label=metric, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{bar.get_height():.3f}', ha='center', fontsize=8)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df.index, rotation=15)
ax.legend(loc='lower right')
ax.set_ylim(0.9, 1.0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('Confusion Matrices for All Models', fontweight='bold', fontsize=14)

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-dep', 'Dep'], yticklabels=['Non-dep', 'Dep'])
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Hyperparameter Tuning

Perform grid search with 5-fold stratified cross-validation on the best-performing model to optimise hyperparameters.

In [ ]:
# ============================================================
# STEP 5: HYPERPARAMETER TUNING
# ============================================================

print("=" * 60)
print(f"HYPERPARAMETER TUNING: {best_model_name}")
print("=" * 60)

if best_model_name == 'Logistic Regression':
    param_grid = {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs']
    }
    base_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
elif best_model_name == 'Linear SVM':
    param_grid = {
        'C': [0.01, 0.1, 1, 10],
        'loss': ['hinge', 'squared_hinge']
    }
    base_model = LinearSVC(max_iter=1000, random_state=42, class_weight='balanced')
else:
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 20, None],
    }
    base_model = RandomForestClassifier(random_state=42, class_weight='balanced')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    base_model, param_grid, cv=cv, scoring='f1_weighted',
    n_jobs=-1, verbose=1, return_train_score=True
)

grid_search.fit(X_train, y_train)

# Results
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)

print(f"\n✅ Best Parameters: {grid_search.best_params_}")
print(f"✅ Best CV F1 Score: {grid_search.best_score_:.4f}")
print(f"\n📊 Test Set Performance (Tuned Model):")
print(f"   Accuracy:  {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"   Precision: {precision_score(y_test, y_pred_tuned, average='weighted'):.4f}")
print(f"   Recall:    {recall_score(y_test, y_pred_tuned, average='weighted'):.4f}")
print(f"   F1-Score:  {f1_score(y_test, y_pred_tuned, average='weighted'):.4f}")

print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred_tuned, target_names=['Non-depressive', 'Depressive']))

In [ ]:
# Visualise CV results
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results.sort_values('rank_test_score')

fig, ax = plt.subplots(figsize=(10, 5))
params_str = [str(p) for p in cv_results['params'][:10]]
ax.barh(params_str, cv_results['mean_test_score'][:10], 
        xerr=cv_results['std_test_score'][:10], color='#3498db', alpha=0.8)
ax.set_xlabel('Mean CV F1 Score')
ax.set_title(f'Hyperparameter Tuning Results ({best_model_name})', fontweight='bold')
ax.set_xlim(0.9, 1.0)

plt.tight_layout()
plt.savefig('tuning_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Model Evaluation & Visualisation

Comprehensive evaluation of the tuned model with confusion matrix, classification metrics, and feature importance analysis.

In [ ]:
# Final confusion matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_tuned)
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn_r', ax=ax1,
            xticklabels=['Non-depressive', 'Depressive'],
            yticklabels=['Non-depressive', 'Depressive'],
            linewidths=2, linecolor='white')
ax1.set_title('Tuned Model - Confusion Matrix', fontweight='bold')
ax1.set_xlabel('Predicted Label')
ax1.set_ylabel('True Label')

# Metrics bar chart
metrics_dict = {
    'Accuracy': accuracy_score(y_test, y_pred_tuned),
    'Precision': precision_score(y_test, y_pred_tuned, average='weighted'),
    'Recall': recall_score(y_test, y_pred_tuned, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_tuned, average='weighted'),
    'CV F1': grid_search.best_score_,
}

bars = ax2.bar(metrics_dict.keys(), metrics_dict.values(), 
               color=['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#9b59b6'],
               edgecolor='white', linewidth=2)
ax2.set_title('Tuned Model - Performance Metrics', fontweight='bold')
ax2.set_ylim(0.9, 1.0)
ax2.set_ylabel('Score')
for bar, val in zip(bars, metrics_dict.values()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('final_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (top words for each class)
if hasattr(tuned_model, 'coef_'):
    feature_names = tfidf.get_feature_names_out()
    coefs = tuned_model.coef_[0]
    
    top_dep = np.argsort(coefs)[-15:][::-1]
    top_nondep = np.argsort(coefs)[:15]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Top depressive indicators
    ax1.barh([feature_names[i] for i in top_dep], 
             [coefs[i] for i in top_dep], color='#e74c3c', alpha=0.8)
    ax1.set_title('Top Depressive Indicator Words', fontweight='bold')
    ax1.set_xlabel('Feature Weight')
    ax1.invert_yaxis()
    
    # Top non-depressive indicators
    ax2.barh([feature_names[i] for i in top_nondep],
             [abs(coefs[i]) for i in top_nondep], color='#2ecc71', alpha=0.8)
    ax2.set_title('Top Non-depressive Indicator Words', fontweight='bold')
    ax2.set_xlabel('Feature Weight (absolute)')
    ax2.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Feature importance not available for this model type.")

## 9. Error Analysis

Examine misclassified samples to understand model limitations and inform ethical assessment.

In [ ]:
# ============================================================
# ERROR ANALYSIS
# ============================================================

print("=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

# Find misclassified samples
test_indices = df.index[y == y_test[0]]  # Approximate
misclassified_mask = y_pred_tuned != y_test

n_errors = misclassified_mask.sum()
print(f"\n❌ Total misclassifications: {n_errors} out of {len(y_test)} ({n_errors/len(y_test)*100:.1f}%)")

# Get test texts
test_df = df.iloc[-len(y_test):].reset_index(drop=True)

fp_mask = (y_pred_tuned == 1) & (y_test == 0)
fn_mask = (y_pred_tuned == 0) & (y_test == 1)

print(f"\n🔴 False Positives (predicted depressive, actually not): {fp_mask.sum()}")
if fp_mask.sum() > 0:
    fp_indices = np.where(fp_mask)[0]
    for idx in fp_indices[:3]:
        print(f"   Text: {test_df.iloc[idx]['cleaned_text'][:100]}...")

print(f"\n🟡 False Negatives (predicted non-depressive, actually depressive): {fn_mask.sum()}")
if fn_mask.sum() > 0:
    fn_indices = np.where(fn_mask)[0]
    for idx in fn_indices[:3]:
        print(f"   Text: {test_df.iloc[idx]['cleaned_text'][:100]}...")

print(f"\n✅ True Positives: {((y_pred_tuned == 1) & (y_test == 1)).sum()}")
print(f"✅ True Negatives: {((y_pred_tuned == 0) & (y_test == 0)).sum()}")

print(f"\n⚠️ Key observations:")
print("  - False negatives are clinically concerning (missed depression cases)")
print("  - False positives may cause unnecessary anxiety or stigma")
print("  - Both types of errors have ethical implications for deployment")

## 10. Ethical Risk Assessment

Evaluate the ethical dimensions of the AI system using established frameworks:
- **Floridi et al. (2018)** — AI4People ethical framework
- **Timmons et al. (2023)** — Bias assessment in mental health AI
- **Wellner & Mykhailov (2024)** — Ethics of care perspective

In [ ]:
# ============================================================
# ETHICAL RISK ASSESSMENT
# ============================================================

print("=" * 60)
print("ETHICAL RISK ASSESSMENT")
print("=" * 60)

ethical_assessment = {
    "Data Privacy & Consent": {
        "status": "✅ LOW RISK",
        "detail": "Dataset is publicly available. No PII processed. User data in Kansi AI stored with hashed passwords."
    },
    "Algorithmic Fairness": {
        "status": "⚠️ MODERATE RISK",
        "detail": "Balanced training data (50/50). However, real-world depression prevalence is ~4%. Model not tested across demographic subgroups."
    },
    "Transparency": {
        "status": "✅ LOW RISK",
        "detail": "Logistic Regression is interpretable. Feature weights can be inspected. Confidence scores provided."
    },
    "Misclassification Risk": {
        "status": "⚠️ MODERATE RISK",
        "detail": f"False negatives: {fn_mask.sum()} (missed depression). False positives: {fp_mask.sum()} (unnecessary concern). Both have clinical consequences."
    },
    "Human Oversight": {
        "status": "✅ IMPLEMENTED",
        "detail": "System designed as screening support tool. Clear disclaimers. Not positioned as diagnostic instrument."
    },
    "Cultural Sensitivity": {
        "status": "⚠️ MODERATE RISK",
        "detail": "Dataset primarily in English. Depression expression varies across cultures. Cross-cultural validation needed."
    },
    "Accountability": {
        "status": "✅ LOW RISK",
        "detail": "All model parameters documented. Code reproducible. Analysis history maintained for audit."
    }
}

for category, info in ethical_assessment.items():
    print(f"\n{'─' * 50}")
    print(f"📌 {category}: {info['status']}")
    print(f"   {info['detail']}")

print(f"\n{'=' * 60}")
print("OVERALL ASSESSMENT: Model is technically sound but requires")
print("additional safeguards before real-world clinical deployment.")
print("Human oversight is essential. Cross-cultural and demographic")
print("bias testing is recommended before scaling.")
print(f"{'=' * 60}")

## 11. Kansi AI Application (SQLite + Flask)

The Kansi AI platform is built with:
- **SQLite** database for user accounts, authentication, and analysis history
- **Flask** web framework for the user interface
- **Google OAuth + Email/Password** authentication
- **Model inference** integrated into the web application

Below is the database setup and a demonstration of the core functionality.

In [ ]:
# ============================================================
# KANSI AI - SQLITE DATABASE SETUP
# ============================================================

DB_PATH = 'kansi_ai.db'

def init_database():
    """Initialise the Kansi AI SQLite database."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Users table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            email TEXT UNIQUE NOT NULL,
            password_hash TEXT,
            full_name TEXT NOT NULL,
            auth_provider TEXT DEFAULT 'email',
            google_id TEXT UNIQUE,
            profile_picture TEXT,
            bio TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            last_login TIMESTAMP,
            is_active BOOLEAN DEFAULT 1
        )
    ''')
    
    # Analysis history table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS analysis_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            input_text TEXT NOT NULL,
            prediction TEXT NOT NULL,
            confidence REAL,
            model_used TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE CASCADE
        )
    ''')
    
    # User preferences
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS user_preferences (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER UNIQUE NOT NULL,
            theme TEXT DEFAULT 'light',
            notifications_enabled BOOLEAN DEFAULT 1,
            FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE CASCADE
        )
    ''')
    
    conn.commit()
    conn.close()
    print("✅ Database initialised successfully!")

def hash_password(password):
    salt = secrets.token_hex(16)
    pwd_hash = hashlib.sha256((salt + password).encode()).hexdigest()
    return f"{salt}:{pwd_hash}"

def verify_password(stored_hash, password):
    salt, pwd_hash = stored_hash.split(':')
    return hashlib.sha256((salt + password).encode()).hexdigest() == pwd_hash

def create_user(email, password, full_name, auth_provider='email'):
    conn = sqlite3.connect(DB_PATH)
    try:
        password_hash = hash_password(password) if password else None
        conn.execute(
            'INSERT INTO users (email, password_hash, full_name, auth_provider) VALUES (?, ?, ?, ?)',
            (email, password_hash, full_name, auth_provider)
        )
        conn.commit()
        user_id = conn.execute('SELECT id FROM users WHERE email = ?', (email,)).fetchone()[0]
        conn.execute('INSERT INTO user_preferences (user_id) VALUES (?)', (user_id,))
        conn.commit()
        return user_id
    except sqlite3.IntegrityError:
        return None
    finally:
        conn.close()

def authenticate_user(email, password):
    conn = sqlite3.connect(DB_PATH)
    row = conn.execute('SELECT id, password_hash, full_name FROM users WHERE email = ?', (email,)).fetchone()
    conn.close()
    if row and row[1] and verify_password(row[1], password):
        return {'id': row[0], 'name': row[2]}
    return None

# Initialise and test
init_database()

# Create test user
user_id = create_user('researcher@kansi.ai', 'SecurePass123!', 'Dr. Test Researcher')
print(f"\n👤 Test user created (ID: {user_id})")

# Authenticate
auth = authenticate_user('researcher@kansi.ai', 'SecurePass123!')
print(f"🔐 Authentication test: {'✅ PASSED' if auth else '❌ FAILED'}")
if auth:
    print(f"   Logged in as: {auth['name']}")

# Google auth simulation
google_user_id = create_user('user@gmail.com', None, 'Google User', auth_provider='google')
print(f"\n🔵 Google OAuth user created (ID: {google_user_id})")

# Show database schema
conn = sqlite3.connect(DB_PATH)
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print(f"\n📋 Database tables: {[t[0] for t in tables]}")
for table in tables:
    cols = conn.execute(f"PRAGMA table_info({table[0]})").fetchall()
    print(f"   {table[0]}: {[c[1] for c in cols]}")
conn.close()

## 12. Model Deployment & Inference

Save the trained model and demonstrate the Kansi AI inference pipeline.

In [ ]:
# ============================================================
# SAVE MODEL ARTIFACTS
# ============================================================

os.makedirs('models', exist_ok=True)

# Save model
joblib.dump(tuned_model, 'models/kansi_ai_model.pkl')
print("✅ Model saved: models/kansi_ai_model.pkl")

# Save TF-IDF vectoriser
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')
print("✅ TF-IDF saved: models/tfidf_vectorizer.pkl")

# Save label encoder
joblib.dump(le, 'models/label_encoder.pkl')
print("✅ Label encoder saved: models/label_encoder.pkl")

# Save training results
training_results = {
    'model_name': best_model_name,
    'training_date': datetime.now().isoformat(),
    'best_params': grid_search.best_params_,
    'cv_f1_score': round(grid_search.best_score_, 4),
    'test_accuracy': round(accuracy_score(y_test, y_pred_tuned), 4),
    'test_f1_score': round(f1_score(y_test, y_pred_tuned, average='weighted'), 4),
    'model_comparison': {name: {k: v for k, v in r.items() if k != 'confusion_matrix'} 
                         for name, r in results.items()}
}

with open('models/training_results.json', 'w') as f:
    json.dump(training_results, f, indent=2)
print("✅ Results saved: models/training_results.json")

In [ ]:
# ============================================================
# KANSI AI INFERENCE DEMO
# ============================================================

print("=" * 60)
print("🧠 KANSI AI - LIVE INFERENCE DEMO")
print("=" * 60)

# Load saved model
loaded_model = joblib.load('models/kansi_ai_model.pkl')
loaded_tfidf = joblib.load('models/tfidf_vectorizer.pkl')

def kansi_ai_predict(text):
    """Kansi AI prediction function."""
    # Clean
    cleaned = text.lower()
    cleaned = re.sub(r'[^a-zA-Z\s]', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    
    # Transform
    features = loaded_tfidf.transform([cleaned])
    prediction = loaded_model.predict(features)[0]
    
    if hasattr(loaded_model, 'predict_proba'):
        proba = loaded_model.predict_proba(features)[0]
        confidence = float(max(proba)) * 100
    else:
        confidence = min(abs(float(loaded_model.decision_function(features)[0])) * 20, 99.0)
    
    label = '🔴 Depressive Indicators Detected' if prediction == 1 else '🟢 No Depressive Indicators'
    return label, round(confidence, 2)

# Test samples
test_texts = [
    "I feel so empty and hopeless, nothing matters anymore and I can't find the energy to do anything",
    "Had an amazing day at the park with my family, feeling grateful and happy",
    "I've been isolating myself from everyone and I can't stop crying",
    "Just got promoted at work and celebrated with friends over dinner!",
    "I can't sleep, my mind keeps racing with dark thoughts every night",
    "Looking forward to my vacation next week, so excited to explore new places",
]

print()
for text in test_texts:
    label, confidence = kansi_ai_predict(text)
    print(f"📝 \"{text[:70]}...\"")
    print(f"   Result: {label} (Confidence: {confidence}%)")
    print()

# Save analysis to database
conn = sqlite3.connect(DB_PATH)
for text in test_texts[:3]:
    label, confidence = kansi_ai_predict(text)
    conn.execute(
        'INSERT INTO analysis_history (user_id, input_text, prediction, confidence, model_used) VALUES (?, ?, ?, ?, ?)',
        (user_id, text[:500], label, confidence, best_model_name)
    )
conn.commit()

# Show history
history = conn.execute('SELECT input_text, prediction, confidence FROM analysis_history WHERE user_id = ?', (user_id,)).fetchall()
print(f"\n📋 Analysis history for user {user_id}:")
for h in history:
    print(f"   [{h[2]}%] {h[1]}: {h[0][:60]}...")
conn.close()

print(f"\n⚠️ DISCLAIMER: Kansi AI is a screening tool, NOT a clinical diagnosis.")
print("   If you are experiencing mental health difficulties, please consult")
print("   a qualified mental health professional.")

## 13. Conclusions & Future Work

### Key Findings
- **Logistic Regression** achieved the highest performance with a **95.83% F1-score** on the test set and **97.37% cross-validated F1-score**
- Logistic Regression and Linear SVM performed equally well, outperforming Random Forest and Gradient Boosting on this dataset
- TF-IDF features effectively captured depressive linguistic markers such as negative emotional tone and self-referential language
- The model demonstrated balanced performance across both classes

### Ethical Considerations
- **Algorithmic bias** risk exists due to limited dataset diversity and cultural representation
- **Misclassification** consequences are significant in mental health contexts — both false positives and false negatives carry ethical weight
- **Human oversight** is essential — the system should supplement, not replace, professional clinical assessment
- **Transparency** is achieved through interpretable model choice and confidence score reporting

### Future Research Directions
1. Expand to larger, more diverse datasets (clinical notes, social media, multilingual text)
2. Incorporate contextual embeddings (BERT, RoBERTa) for nuanced language understanding
3. Conduct cross-cultural and demographic bias audits
4. Implement explainable AI (XAI) methods for clinical transparency
5. Longitudinal studies on real-world screening impact
6. Participatory design research with mental health professionals and service users

### Kansi AI Platform
The Kansi AI web application demonstrates responsible deployment with:
- Secure user authentication (email + Google OAuth)
- SQLite database for data management
- Analysis history and confidence scoring
- Clear disclaimers and pathways to professional support

---
*This notebook was developed as part of the Kansi AI research project investigating AI-based text analysis for mental health screening.*